In [1]:
from chunking import *
import torch
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
import math

/Users/prateekM/Desktop/1_Projects/Project Chitti/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [3]:
def get_embeddings(chunks, batch_size=8):
    return model.encode(chunks, convert_to_tensor=True, show_progress_bar=True, batch_size=batch_size)

In [4]:
get_embeddings(["Hi, my name is Prateek", "Hi, my name is Prateek", "Hi, my name is Prateek", "Hi, my name is Prateek", "Hi, my name is Prateek", "Hi, my name is Prateek", "Hi, my name is Prateek", "Hi, my name is Prateek"])

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.55it/s]


tensor([[-0.0850, -0.0042, -0.0685,  ...,  0.0153, -0.0497,  0.0190],
        [-0.0850, -0.0042, -0.0685,  ...,  0.0153, -0.0497,  0.0190],
        [-0.0850, -0.0042, -0.0685,  ...,  0.0153, -0.0497,  0.0190],
        ...,
        [-0.0850, -0.0042, -0.0685,  ...,  0.0153, -0.0497,  0.0190],
        [-0.0850, -0.0042, -0.0685,  ...,  0.0153, -0.0497,  0.0190],
        [-0.0850, -0.0042, -0.0685,  ...,  0.0153, -0.0497,  0.0190]],
       device='mps:0')

In [5]:
chunks = [
    '''The story begins in Springfield, Massachusetts, 1891. Dr. James Naismith's simple invention
of nailing peach baskets to a gymnasium wall has evolved into something extraordinary.
Today's basketball landscape showcases a sport that has broken free from its American
roots, becoming a cultural force that bridges continents and cultures. This transformation
reflects not just athletic evolution, but a broader story of global connectivity and shared
passion''',
    '''The NBA's transformation tells a compelling story of basketball's globalization. Gone are the
days when the league was predominantly American. Today's NBA features transcendent
international talents like Nikola Jokić, Joel Embiid, and Giannis Antetokounmpo – players
who have redefined excellence in the sport. Their success represents more than individual
achievement; it symbolizes basketball's power to discover and nurture talent regardless of
origin''',
    '''The statistics paint a vivid picture of basketball's global reach. FIBA's latest reports indicate
that over 450 million people actively play basketball worldwide. The NBA's global broadcast
reaches 215 countries and territories in 47 languages. In the 2023-24 season, 125
international players from 40 countries graced NBA rosters on opening night. Perhaps most
strikingly, China alone boasts 300 million basketball players – a number that exceeds the
entire U.S. population.''',
    '''The women's game has written its own remarkable chapter in basketball's global story. The
WNBA continues to expand its international influence, with stars like Jonquel Jones and Ezi
Magbegor leading the charge. As WNBA Commissioner Cathy Engelbert notes, "Women's
basketball is experiencing unprecedented growth." This growth manifests in increased
viewership, engagement, and participation across demographics, creating new role models
for aspiring female athletes worldwide'''
]

In [6]:
chunk_embedding = get_embeddings(chunks=chunks)

Batches: 100%|██████████| 1/1 [00:00<00:00, 11.72it/s]


In [7]:
user_input = 'Where does the story begin?'
question_embedding = get_embeddings(chunks=user_input)

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.38it/s]


In [8]:
def cosine_sim(a, b):
    if len(a) != len(b):
        raise ValueError("Embedding lengths do not match")
    
    dot_value = 0
    magnitude_a = 0
    magnitude_b = 0

    for i, j in zip(a, b):
        dot_value += i * j
        magnitude_a += i**2
        magnitude_b += j**2
    
    return dot_value / (magnitude_a**0.5 * magnitude_b**0.5)

In [9]:
cosine_sim(a=[1, 2, 3], b=[4, 5, 6])

0.9746318461970762

In [10]:
embedding_sim_score: dict = {c+1: cosine_sim(ce, question_embedding).item() for c, ce in enumerate(chunk_embedding)}
embedding_sim_score

{1: 0.342067152261734,
 2: 0.09772597998380661,
 3: -0.009490849450230598,
 4: 0.16974422335624695}

In [11]:
relevent_chunk = chunks[max(embedding_sim_score, key=embedding_sim_score.get)]

In [12]:
PROMPT = '''
You are a helpful teaching assistant. Your task is to answer the user's question given the context of the information. 
Always use the information given to you to answer the question, and do not make anything up. 

Question: {}

Context: {}
'''

In [13]:
USER_PROMPT = PROMPT.format(user_input, relevent_chunk)

In [14]:
print(USER_PROMPT)


You are a helpful teaching assistant. Your task is to answer the user's question given the context of the information. 
Always use the information given to you to answer the question, and do not make anything up. 

Question: Where does the story begin?

Context: The NBA's transformation tells a compelling story of basketball's globalization. Gone are the
days when the league was predominantly American. Today's NBA features transcendent
international talents like Nikola Jokić, Joel Embiid, and Giannis Antetokounmpo – players
who have redefined excellence in the sport. Their success represents more than individual
achievement; it symbolizes basketball's power to discover and nurture talent regardless of
origin

